In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.utils import shuffle
from scipy.stats import uniform

In [2]:
# 가상의 데이터셋 생성 (CIFAR-10과 유사한 데이터셋)
num_classes = 10
input_shape = (32, 32, 3)
num_samples = 1000

X = np.random.rand(num_samples, *input_shape)
y = np.random.randint(num_classes, size=num_samples)

In [3]:
# 데이터셋 셔플 및 분할
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# === 각 하이퍼파라미터 trial 시작 시 가장 먼저 호출 ===
K.clear_session()

def create_model(learning_rate=0.001):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False          # 전이학습 안정화 (가장 중요)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    opt = Adam(learning_rate=learning_rate, clipnorm=1.0)

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [5]:
# 하이퍼파라미터 분포 정의
param_dist = {
    'learning_rate': uniform(0.001, 0.1),
    'batch_size': [16, 32, 64]
}

best_accuracy = 0
best_params = {}
n_iter = 5

# 랜덤하게 하이퍼파라미터 조합을 선택하여 시도
for params in ParameterSampler(param_dist, n_iter=n_iter, random_state=42):
    model = create_model(learning_rate=params['learning_rate'])
    model.fit(X_train, y_train, epochs=10, batch_size=params['batch_size'], validation_split=0.2, verbose=0)

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Params: {params}, Test Accuracy: {accuracy}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = params

print("Best Parameters (RandomSearch):", best_params)
print("Best Test Accuracy (RandomSearch):", best_accuracy)

Params: {'batch_size': 64, 'learning_rate': np.float64(0.08065429868602329)}, Test Accuracy: 0.11999999731779099
Params: {'batch_size': 64, 'learning_rate': np.float64(0.07419939418114051)}, Test Accuracy: 0.11999999731779099
Params: {'batch_size': 16, 'learning_rate': np.float64(0.0606850157946487)}, Test Accuracy: 0.125
Params: {'batch_size': 32, 'learning_rate': np.float64(0.016599452033620267)}, Test Accuracy: 0.11999999731779099
Params: {'batch_size': 64, 'learning_rate': np.float64(0.04692488919658672)}, Test Accuracy: 0.11999999731779099
Best Parameters (RandomSearch): {'batch_size': 16, 'learning_rate': np.float64(0.0606850157946487)}
Best Test Accuracy (RandomSearch): 0.125


In [6]:
# 최적의 하이퍼파라미터로 모델 재학습
best_model = create_model(learning_rate=best_params['learning_rate'])
best_model.fit(X_train, y_train, epochs=10, batch_size=best_params['batch_size'], validation_split=0.2)

# 테스트 데이터셋으로 모델 평가
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 137ms/step - accuracy: 0.0844 - loss: 68.8894 - val_accuracy: 0.1063 - val_loss: 2.3067
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1187 - loss: 2.3046 - val_accuracy: 0.0688 - val_loss: 2.3081
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1063 - loss: 2.3071 - val_accuracy: 0.0688 - val_loss: 2.3109
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1125 - loss: 2.3117 - val_accuracy: 0.0688 - val_loss: 2.3024
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1016 - loss: 2.3087 - val_accuracy: 0.0688 - val_loss: 2.3135
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1187 - loss: 2.3083 - val_accuracy: 0.0688 - val_loss: 2.3068
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1047 - loss: 2.3120 - val_accuracy: 0.0688 - val_loss: 2.3182
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1000 - loss: 2.3095 - val_accuracy: 0.1187 - val